# International Football Results Analysis

This notebook analyses international football match results from 1872 to 2026 using Python and Pandas.

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for saving files without display
import matplotlib.pyplot as plt
import numpy as np

# Load the dataset
df = pd.read_csv("results.csv")
df.head()

---
## Basic Exploration

### 1. How many matches are in the dataset?

Each row in the dataset represents one match. We use `len()` on the DataFrame to count the total number of rows, which equals the total number of matches.

In [ ]:
num_matches = len(df)
print(f"Total number of matches: {num_matches}")

### 2. What is the earliest and latest year in the data?

The `date` column contains match dates as strings. We convert it to a proper `datetime` type using `pd.to_datetime()`, then extract the year with `.dt.year`. Finally, `.min()` and `.max()` give us the range.

In [ ]:
df['date'] = pd.to_datetime(df['date'])
earliest_year = df['date'].dt.year.min()
latest_year = df['date'].dt.year.max()
print(f"Earliest year: {earliest_year}")
print(f"Latest year:   {latest_year}")

### 3. How many unique countries are there?

Countries appear in both the `home_team` and `away_team` columns. We concatenate both columns into a single Series using `pd.concat()`, then call `.unique()` to get distinct values, and `len()` to count them.

In [ ]:
all_countries = pd.concat([df['home_team'], df['away_team']]).unique()
num_unique_countries = len(all_countries)
print(f"Number of unique countries: {num_unique_countries}")

### 4. Which team appears most frequently as home team?

`value_counts()` on the `home_team` column returns a Series sorted by frequency (descending). `idxmax()` returns the index (team name) of the highest count, and `max()` returns the count itself.

In [ ]:
home_counts = df['home_team'].value_counts()
most_frequent_home = home_counts.idxmax()
home_count = home_counts.max()
print(f"Most frequent home team: {most_frequent_home} ({home_count} matches)")

---
## Goals Analysis

First we create a `total_goals` column by adding `home_score` and `away_score` together. This makes it easy to answer questions about overall scoring.

In [ ]:
df["total_goals"] = df["home_score"] + df["away_score"]
df[["home_team", "away_team", "home_score", "away_score", "total_goals"]].head()

### 5. What is the average number of goals per match?

`.mean()` on the `total_goals` column calculates the arithmetic average across all matches.

In [ ]:
avg_goals = df["total_goals"].mean()
print(f"Average goals per match: {avg_goals:.2f}")

### 6. What is the highest scoring match?

`idxmax()` returns the row index where `total_goals` is highest. We then use `df.loc[]` to retrieve the full row and display all the match details.

In [ ]:
highest_idx = df["total_goals"].idxmax()
highest_match = df.loc[highest_idx]
print(f"Highest scoring match: {highest_match['home_team']} vs {highest_match['away_team']}")
print(f"Score: {int(highest_match['home_score'])}-{int(highest_match['away_score'])} ({int(highest_match['total_goals'])} goals) on {highest_match['date'].strftime('%Y-%m-%d')}")

### 7. Are more goals scored at home or away?

We sum the `home_score` column and the `away_score` column separately using `.sum()`, then compare the two totals.

In [ ]:
total_home_goals = df["home_score"].sum()
total_away_goals = df["away_score"].sum()
print(f"Total home goals: {total_home_goals}")
print(f"Total away goals: {total_away_goals}")
print(f"More goals scored: {'Home' if total_home_goals > total_away_goals else 'Away'}")

### 8. What is the most common total goals value?

`value_counts()` on `total_goals` counts how often each score total appears. `idxmax()` returns the value that occurs most often (the statistical **mode**).

In [ ]:
goals_freq = df["total_goals"].value_counts()
most_common_goals = goals_freq.idxmax()
most_common_count = goals_freq.max()
print(f"Most common total goals: {int(most_common_goals)} (occurred in {most_common_count} matches)")

---
## Match Results Analysis

We classify each match as a **Home Win**, **Away Win**, or **Draw** by comparing `home_score` and `away_score`. A helper function `match_result` encodes this logic and is applied row-by-row with `df.apply()`.

In [ ]:
def match_result(row):
    if row["home_score"] > row["away_score"]:
        return "Home Win"
    elif row["home_score"] < row["away_score"]:
        return "Away Win"
    else:
        return "Draw"

df["result"] = df.apply(match_result, axis=1)
df["result"].value_counts()

### 9. What percentage of matches are home wins?

`value_counts(normalize=True)` returns proportions (values between 0 and 1). Multiplying by 100 converts to percentages. We index into the result with `"Home Win"` to get the home-win percentage.

In [ ]:
result_counts = df["result"].value_counts()
result_percentages = df["result"].value_counts(normalize=True) * 100
home_win_pct = result_percentages["Home Win"]
print(f"Home win percentage: {home_win_pct:.2f}%")
print(f"Away win percentage: {result_percentages['Away Win']:.2f}%")
print(f"Draw percentage:     {result_percentages['Draw']:.2f}%")

### 10. Does home advantage exist?

Home advantage exists if home teams win significantly more often than away teams. We look at two pieces of evidence:
1. The percentage of home wins vs away wins.
2. The total goals scored at home vs away.

If both metrics favour the home side, we can conclude home advantage is real.

In [ ]:
print("Does home advantage exist? YES")
print(f"  Evidence 1 — Win %: Home ({result_percentages['Home Win']:.1f}%) > Away ({result_percentages['Away Win']:.1f}%)")
print(f"  Evidence 2 — Goals: Home ({total_home_goals}) > Away ({total_away_goals})")
print(f"  Home teams win {result_percentages['Home Win']/result_percentages['Away Win']:.2f}x more often than away teams")

### 11. Which country has the most wins historically?

For every non-draw match we record the winning team in a new `winner` column using another helper function. We then call `value_counts()` on `winner` to rank countries by total wins.

In [ ]:
def get_winner(row):
    if row["result"] == "Home Win":
        return row["home_team"]
    elif row["result"] == "Away Win":
        return row["away_team"]
    else:
        return None  # Draw — no winner

df["winner"] = df.apply(get_winner, axis=1)

wins_by_country = df["winner"].value_counts()
top_winner = wins_by_country.idxmax()
top_wins = wins_by_country.max()
print(f"Country with most wins: {top_winner} ({top_wins} wins)")
print("\nTop 5 countries by wins:")
print(wins_by_country.head(5).to_string())

---
## Visualizations

### Histogram of Goals Per Match

`hist()` on `total_goals` shows the frequency distribution of scoring across all matches. Using `bins=15` divides the range into 15 equal intervals, giving enough detail without being noisy.

In [ ]:
plt.figure(figsize=(10, 6))
df["total_goals"].hist(bins=15, edgecolor='black', color='steelblue')
plt.title("Distribution of Goals Per Match", fontsize=14)
plt.xlabel("Total Goals", fontsize=12)
plt.ylabel("Number of Matches", fontsize=12)
plt.grid(axis='y', alpha=0.7)
plt.tight_layout()
plt.savefig("histogram_goals.png", dpi=150)
plt.show()
plt.close()

### Bar Chart of Match Outcomes

We use `value_counts()` to get the count for each outcome category and then `.plot(kind='bar')`. Different colours make it easy to distinguish Home Win, Away Win, and Draw at a glance. Value labels are added on top of each bar for precision.

In [ ]:
plt.figure(figsize=(8, 6))
colors = ['forestgreen', 'coral', 'gray']  # Home Win, Away Win, Draw
result_counts.plot(kind='bar', color=colors, edgecolor='black')
plt.title("Match Outcomes Distribution", fontsize=14)
plt.xlabel("Result", fontsize=12)
plt.ylabel("Number of Matches", fontsize=12)
plt.xticks(rotation=0)
for i, v in enumerate(result_counts):
    plt.text(i, v + 200, str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.savefig("bar_match_outcomes.png", dpi=150)
plt.show()
plt.close()

### Top 10 Teams by Total Wins

We take the top 10 entries from the `wins_by_country` Series and plot a **horizontal** bar chart. Sorting the values before plotting ensures the country with the most wins appears at the top, making the chart easy to read.

In [ ]:
plt.figure(figsize=(10, 6))
top_10_wins = wins_by_country.head(10)
top_10_wins.sort_values().plot(kind='barh', color='royalblue', edgecolor='black')
plt.title("Top 10 Countries by Total Wins", fontsize=14)
plt.xlabel("Number of Wins", fontsize=12)
plt.ylabel("Country", fontsize=12)
for i, v in enumerate(top_10_wins.sort_values()):
    plt.text(v + 5, i, str(v), va='center', fontsize=10)
plt.tight_layout()
plt.savefig("top10_wins.png", dpi=150)
plt.show()
plt.close()

print("Visualizations saved: histogram_goals.png, bar_match_outcomes.png, top10_wins.png")